# Logistic regression — pure NumPy

**Dataset:** `breast_cancer.csv` (569 rows × 30 numeric features, binary `diagnosis`). `0 = malignant`, `1 = benign`.

**Goal:** probability model, honest split, and an operating threshold you can defend.

## 1. Load and inspect the label balance

In [ ]:
import numpy as np

raw = np.genfromtxt('breast_cancer.csv', delimiter=',', names=True)
feature_names = list(raw.dtype.names[:-1])
X = np.column_stack([raw[name] for name in feature_names]).astype(float)
y = np.asarray(raw['diagnosis'], dtype=int)

print('rows:', X.shape[0], '| features:', X.shape[1])
print('positive (benign):', int(y.sum()), '| negative (malignant):', int((y == 0).sum()))
print('positive rate:', round(float(y.mean()), 3))

## 2. Stratified split — keep the class balance in both halves

In [ ]:
rng = np.random.default_rng(11)
train_idx, test_idx = [], []
for label in (0, 1):
    idx = np.where(y == label)[0]
    rng.shuffle(idx)
    cut = int(0.8 * len(idx))
    train_idx.extend(idx[:cut])
    test_idx.extend(idx[cut:])
train_idx = np.array(train_idx)
test_idx = np.array(test_idx)

print('train:', len(train_idx), '| positives:', int(y[train_idx].sum()))
print('test :', len(test_idx), '| positives:', int(y[test_idx].sum()))

## 3. Sigmoid and gradient descent

In [ ]:
mean = X[train_idx].mean(axis=0)
scale = X[train_idx].std(axis=0)
scale[scale == 0] = 1.0
Xs = (X - mean) / scale

Xtr = np.column_stack([np.ones(len(train_idx)), Xs[train_idx]])
Xte = np.column_stack([np.ones(len(test_idx)), Xs[test_idx]])
ytr, yte = y[train_idx], y[test_idx]

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

weights = np.zeros(Xtr.shape[1])
learning_rate = 0.1
l2 = 1e-3
for epoch in range(2500):
    p = sigmoid(Xtr @ weights)
    gradient = (Xtr.T @ (p - ytr)) / len(ytr)
    gradient[1:] += l2 * weights[1:]
    weights -= learning_rate * gradient

prob = sigmoid(Xte @ weights)
print('intercept:', round(float(weights[0]), 3))
print('probability range:', round(float(prob.min()), 3), 'to', round(float(prob.max()), 3))

## 4. Confusion matrix and class metrics

In [ ]:
def classification_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (tp + tn) / len(y_true)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}

m = classification_metrics(yte, prob)
for key, value in m.items():
    print(f'{key:>9}: {round(value, 3) if isinstance(value, float) else value}')

## 5. Threshold sweep — the model gives probabilities, you choose the action

In [ ]:
for threshold in (0.2, 0.35, 0.5, 0.65, 0.8):
    m = classification_metrics(yte, prob, threshold)
    print(f't={threshold:.2f} | precision={m["precision"]:.2f} | recall={m["recall"]:.2f} | flagged={m["tp"] + m["fp"]}')

## 6. Which features moved the decision most?

In [ ]:
order = np.argsort(np.abs(weights[1:]))[::-1][:6]
for index in order:
    print(f'{feature_names[index]:>28}: {weights[index + 1]: .3f}')

## 7. Read this before shipping

- Class 1 is **benign** here. In a clinical screen, missing a malignant case (false negative) is usually the expensive error, so do not optimise accuracy alone.
- The split is random; a real evaluation should hold out an entire site or time period.
- Logistic regression is linear in the log-odds. Interactions and non-linear effects need explicit features or another model.
- Probabilities are only calibrated if the data and prevalence match the deployment population. Check calibration before using them as risk.

**Exercise:** lower the threshold until recall for class 0 (malignant) is at least 0.95, then report the false-positive workload.